# 統計モデリングの「分布へフィットしていく動き」をGIFで可視化するノート

"
    "- **目的**: オープンデータを使って、統計モデルがデータ分布や散布図へ段階的にフィットしていく様子を GIF として保存する。
"
    "- **作成環境**: `D:/dev/dev_statistics/venv` の venv kernel (`Python (dev_statistics venv)`)。
"
    "- **作成物**: `outputs/gifs/*.gif` に 4 本のアニメーションを保存する。
"
    "- **注意**: ここでは学習・説明用途として、ブートストラップや正則化付き最小二乗で不確実性を軽量に近似する。本格的なベイズ推論では PyMC/Stan などで事後予測分布をサンプリングする。
"


## 0. 統計モデリングに使いやすいオープンデータ候補

今回の GIF 作成では、軽量で再現しやすい `seaborn-data` の公開 CSV を **ローカル保存済みデータ** から読み込む。ノートブック実行時にインターネットへ取りに行かない。

| データ | ローカルCSV | 使えそうなモデリング | 今回の使用 |
|---|---|---|---|
| Palmer Penguins | [`data/raw/penguins.csv`](../data/raw/penguins.csv) | 体重・くちばし・フリッパー長、種別カテゴリ。単峰分布、単回帰、カテゴリ別回帰、部分プーリングの説明に向く | 1, 2, 4 |
| Old Faithful geyser | [`data/raw/geyser.csv`](../data/raw/geyser.csv) | 噴出時間 `duration` が短い噴出/長い噴出で多峰性になりやすい。混合正規モデルの説明に向く | 3 |
| Tips | [`data/raw/tips.csv`](../data/raw/tips.csv) | `total_bill` と `tip` の単回帰、曜日・性別・喫煙有無でカテゴリあり回帰 | 候補 |
| Auto MPG | [`data/raw/mpg.csv`](../data/raw/mpg.csv) | `horsepower` と `mpg`、`origin` や `cylinders` 別の回帰、非線形性の説明 | 候補 |
| Diamonds | 未保存（今回のGIFでは未使用） | 価格・カラットの歪んだ分布、カテゴリ別価格、対数変換 | 候補 |

元データの出典 URL は [`data/SOURCES.md`](../data/SOURCES.md) にまとめた。

以下の 4 段階でレベルアップする。

1. **棒グラフ + 分布フィッティング**: 体重分布に正規分布を当て、データが増えるほど予測分布が安定する様子。
2. **散布図 + 線形回帰**: フリッパー長から体重を予測し、回帰線と不確実性帯が絞られる様子。
3. **多峰性 + 混合モデル**: 間欠泉の噴出時間を 2 成分混合正規で EM 的にフィットしていく様子。
4. **カテゴリ + 部分プーリング回帰**: 種別ごとの回帰線を、全体平均からカテゴリ別線へ部分的にずらす様子。


In [1]:
# 1. Imports / paths / theme
from __future__ import annotations

import math
import textwrap
import warnings
from pathlib import Path
import imageio.v2 as imageio
import japanize_matplotlib
import matplotlib

matplotlib.use("Agg")  # GIF generation is file-based and restartable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import norm

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
japanize_matplotlib.japanize()  # seaborn が font.family を上書きするため、テーマ設定後に再適用
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 120
plt.rcParams["axes.unicode_minus"] = False

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
GIF_DIR = PROJECT_ROOT / "outputs" / "gifs"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
for path in [DATA_DIR, GIF_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(20260620)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"GIF_DIR      = {GIF_DIR}")


PROJECT_ROOT = D:\dev\dev_statistics
GIF_DIR      = D:\dev\dev_statistics\outputs\gifs


In [2]:
# 2. Load locally saved open data
DATA_FILES = {
    "penguins": DATA_DIR / "penguins.csv",
    "geyser": DATA_DIR / "geyser.csv",
    "tips": DATA_DIR / "tips.csv",
    "mpg": DATA_DIR / "mpg.csv",
}


def load_local_csv(name: str) -> pd.DataFrame:
    path = DATA_FILES[name]
    if not path.exists():
        raise FileNotFoundError(
            f"{path} が見つかりません。公開データは事前にローカル保存してから実行してください。"
        )
    print(f"{name:<8} <- {path.relative_to(PROJECT_ROOT).as_posix()}")
    return pd.read_csv(path)


penguins = load_local_csv("penguins")
geyser = load_local_csv("geyser")
tips = load_local_csv("tips")
mpg = load_local_csv("mpg")

print("penguins", penguins.shape, list(penguins.columns))
print("geyser  ", geyser.shape, list(geyser.columns))
print("tips    ", tips.shape, list(tips.columns))
print("mpg     ", mpg.shape, list(mpg.columns))


penguins <- data/raw/penguins.csv
geyser   <- data/raw/geyser.csv
tips     <- data/raw/tips.csv
mpg      <- data/raw/mpg.csv
penguins (344, 7) ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex']
geyser   (272, 3) ['duration', 'waiting', 'kind']
tips     (244, 7) ['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size']
mpg      (398, 9) ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin', 'name']


In [3]:
# 3. Shared helper functions
def fig_to_frame(fig: matplotlib.figure.Figure) -> np.ndarray:
    """Convert a Matplotlib figure to an RGB numpy frame for imageio."""
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba())
    return rgba[:, :, :3].copy()


def save_gif(frames: list[np.ndarray], path: Path, fps: int = 8) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    imageio.mimsave(path, frames, duration=1 / fps, loop=0)
    print(f"saved {path} ({len(frames)} frames, {fps} fps)")
    return path


def add_note(ax: plt.Axes, text: str, loc: tuple[float, float] = (0.02, 0.98)) -> None:
    ax.text(
        loc[0],
        loc[1],
        text,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=9,
        bbox={"boxstyle": "round,pad=0.35", "fc": "white", "ec": "0.75", "alpha": 0.9},
    )


def normal_bootstrap_density_band(
    x_obs: np.ndarray,
    grid: np.ndarray,
    n_boot: int = 80,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Bootstrap uncertainty band for a Normal density fitted by MLE."""
    rng = np.random.default_rng(seed)
    x_obs = np.asarray(x_obs, dtype=float)
    n = len(x_obs)
    if n < 3:
        mu = x_obs.mean()
        sigma = max(x_obs.std(ddof=1), 1e-3)
        pdf = norm.pdf(grid, mu, sigma)
        return pdf, pdf, pdf
    idx = rng.integers(0, n, size=(n_boot, n))
    samples = x_obs[idx]
    mus = samples.mean(axis=1)
    sigmas = samples.std(axis=1, ddof=1)
    sigmas = np.clip(sigmas, 1e-3, None)
    pdfs = norm.pdf(grid[None, :], loc=mus[:, None], scale=sigmas[:, None])
    return np.quantile(pdfs, 0.05, axis=0), np.mean(pdfs, axis=0), np.quantile(pdfs, 0.95, axis=0)


def fit_ols(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    X = np.column_stack([np.ones_like(x), x])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return beta


def ols_bootstrap_prediction_band(
    x_obs: np.ndarray,
    y_obs: np.ndarray,
    x_grid: np.ndarray,
    n_boot: int = 80,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Bootstrap band for the regression mean line."""
    rng = np.random.default_rng(seed)
    x_obs = np.asarray(x_obs, dtype=float)
    y_obs = np.asarray(y_obs, dtype=float)
    n = len(x_obs)
    preds: list[np.ndarray] = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        beta = fit_ols(x_obs[idx], y_obs[idx])
        preds.append(beta[0] + beta[1] * x_grid)
    arr = np.vstack(preds)
    return np.quantile(arr, 0.05, axis=0), np.mean(arr, axis=0), np.quantile(arr, 0.95, axis=0)


## 1. 棒グラフに対する分布フィッティング

"
    "- **データ**: Palmer Penguins の `body_mass_g`。
"
    "- **モデル**: 単一の正規分布 $y \sim \mathcal{N}(\mu, \sigma^2)$。
"
    "- **動き**: 使う観測数を少しずつ増やし、推定された密度がヒストグラムに近づく様子を描く。
"
    "- **不確実性**: 各時点の観測データをブートストラップし、正規密度の 90% 帯を塗る。
"


In [4]:
def make_normal_fit_gif() -> Path:
    data = penguins["body_mass_g"].dropna().to_numpy(float)
    rng = np.random.default_rng(11)
    data_perm = rng.permutation(data)

    x_min, x_max = data.min() - 250, data.max() + 250
    grid = np.linspace(x_min, x_max, 360)
    bins = np.linspace(x_min, x_max, 24)
    ns = np.unique(np.geomspace(8, len(data), 34).astype(int))

    frames: list[np.ndarray] = []
    y_top = None
    for i, n in enumerate(ns):
        obs = data_perm[:n]
        mu = obs.mean()
        sigma = max(obs.std(ddof=1), 1e-3)
        lo, mid, hi = normal_bootstrap_density_band(obs, grid, n_boot=90, seed=1000 + i)

        fig, ax = plt.subplots(figsize=(8.5, 5.2), constrained_layout=True)
        ax.hist(data, bins=bins, density=True, color="0.78", alpha=0.35, label="全データのヒストグラム")
        ax.hist(obs, bins=bins, density=True, color="#4C72B0", alpha=0.50, label="ここまで観測したデータ")
        ax.fill_between(grid, lo, hi, color="#DD8452", alpha=0.22, label="90% ブートストラップ帯")
        ax.plot(grid, norm.pdf(grid, mu, sigma), color="#C44E52", lw=2.5, label="推定正規分布")
        ax.set_title("1. 体重分布への正規分布フィッティング")
        ax.set_xlabel("body_mass_g")
        ax.set_ylabel("density")
        ax.set_xlim(x_min, x_max)
        if y_top is None:
            y_top = max(hi.max(), 0.001) * 1.28
        ax.set_ylim(0, y_top)
        add_note(
            ax,
            f"観測数 n={n}/{len(data)}\nμ={mu:,.0f} g, σ={sigma:,.0f} g\n少数データでは帯が広く、増えるほど安定",
        )
        ax.legend(loc="upper right", frameon=True)
        frames.append(fig_to_frame(fig))
        plt.close(fig)

    return save_gif(frames, GIF_DIR / "01_normal_distribution_fit.gif", fps=8)


gif_01 = make_normal_fit_gif()
gif_01


saved D:\dev\dev_statistics\outputs\gifs\01_normal_distribution_fit.gif (33 frames, 8 fps)


WindowsPath('D:/dev/dev_statistics/outputs/gifs/01_normal_distribution_fit.gif')

## 2. 散布図に対する線形回帰

"
    "- **データ**: Palmer Penguins の `flipper_length_mm` と `body_mass_g`。
"
    "- **モデル**: $\text{body_mass}_i = \alpha + \beta \cdot \text{flipper}_i + \epsilon_i$。
"
    "- **動き**: 観測点が増えるほど回帰線が安定し、不確実性帯が狭くなる。
"
    "- **不確実性**: 観測済みデータのブートストラップで回帰平均線の 90% 帯を作る。
"


In [5]:
def make_linear_regression_gif() -> Path:
    df = penguins[["flipper_length_mm", "body_mass_g", "species"]].dropna().copy()
    x_all = df["flipper_length_mm"].to_numpy(float)
    y_all = df["body_mass_g"].to_numpy(float)
    rng = np.random.default_rng(22)
    order = rng.permutation(len(df))
    x_perm = x_all[order]
    y_perm = y_all[order]

    x_grid = np.linspace(x_all.min() - 3, x_all.max() + 3, 220)
    ns = np.unique(np.geomspace(12, len(df), 34).astype(int))

    frames: list[np.ndarray] = []
    for i, n in enumerate(ns):
        x_obs = x_perm[:n]
        y_obs = y_perm[:n]
        beta = fit_ols(x_obs, y_obs)
        y_hat = beta[0] + beta[1] * x_grid
        lo, mid, hi = ols_bootstrap_prediction_band(x_obs, y_obs, x_grid, n_boot=90, seed=2000 + i)

        fig, ax = plt.subplots(figsize=(8.5, 5.2), constrained_layout=True)
        ax.scatter(x_all, y_all, s=25, color="0.78", alpha=0.35, label="全データ")
        ax.scatter(x_obs, y_obs, s=34, color="#4C72B0", alpha=0.75, label="ここまで観測")
        ax.fill_between(x_grid, lo, hi, color="#55A868", alpha=0.22, label="90% ブートストラップ帯")
        ax.plot(x_grid, y_hat, color="#2C7F4F", lw=2.6, label="推定回帰線")
        ax.set_title("2. 散布図への単回帰フィッティング")
        ax.set_xlabel("flipper_length_mm")
        ax.set_ylabel("body_mass_g")
        ax.set_xlim(x_grid.min(), x_grid.max())
        ax.set_ylim(y_all.min() - 350, y_all.max() + 350)
        add_note(
            ax,
            f"観測数 n={n}/{len(df)}\ny = {beta[0]:,.0f} + {beta[1]:.1f} x\nデータが増えるほど回帰線の帯が狭い",
        )
        ax.legend(loc="upper left", frameon=True)
        frames.append(fig_to_frame(fig))
        plt.close(fig)

    return save_gif(frames, GIF_DIR / "02_linear_regression_fit.gif", fps=8)


gif_02 = make_linear_regression_gif()
gif_02


saved D:\dev\dev_statistics\outputs\gifs\02_linear_regression_fit.gif (34 frames, 8 fps)


WindowsPath('D:/dev/dev_statistics/outputs/gifs/02_linear_regression_fit.gif')

## 3. 多峰性分布に対する混合モデル

"
    "- **データ**: Old Faithful geyser の噴出時間 `duration`。短い噴出と長い噴出で山が分かれる。
"
    "- **モデル**: 2 成分混合正規 $p(y)=\pi_1\mathcal{N}(\mu_1,\sigma_1^2)+\pi_2\mathcal{N}(\mu_2,\sigma_2^2)$。
"
    "- **動き**: EM アルゴリズムの反復で、成分平均・分散・混合比が多峰性の山に寄っていく様子。
"
    "- **不確実性**: データをブートストラップして、EM の各 iteration 時点で推定される混合密度の 90% 帯を描く。
"


In [6]:
def mixture_pdf(grid: np.ndarray, weights: np.ndarray, means: np.ndarray, sigmas: np.ndarray) -> np.ndarray:
    pdf = np.zeros_like(grid, dtype=float)
    for w, m, s in zip(weights, means, sigmas):
        pdf += w * norm.pdf(grid, m, s)
    return pdf


def em_gmm_1d_trace(x: np.ndarray, n_iter: int = 28) -> list[tuple[np.ndarray, np.ndarray, np.ndarray]]:
    x = np.asarray(x, dtype=float)
    # 少し粗い初期値から始めて、山へ移動する様子が見えるようにする。
    q20, q55 = np.quantile(x, [0.20, 0.55])
    weights = np.array([0.50, 0.50], dtype=float)
    means = np.array([q20, q55], dtype=float)
    sigmas = np.array([x.std(ddof=1) * 1.05, x.std(ddof=1) * 1.05], dtype=float)

    trace: list[tuple[np.ndarray, np.ndarray, np.ndarray]] = []
    for _ in range(n_iter + 1):
        order = np.argsort(means)
        trace.append((weights[order].copy(), means[order].copy(), sigmas[order].copy()))

        component = np.column_stack([weights[k] * norm.pdf(x, means[k], max(sigmas[k], 1e-4)) for k in range(2)])
        denom = np.clip(component.sum(axis=1, keepdims=True), 1e-12, None)
        resp = component / denom
        nk = np.clip(resp.sum(axis=0), 1e-8, None)
        weights = nk / len(x)
        means = (resp * x[:, None]).sum(axis=0) / nk
        var = (resp * (x[:, None] - means) ** 2).sum(axis=0) / nk
        sigmas = np.sqrt(np.clip(var, 1e-4, None))
    return trace


def gmm_bootstrap_trace_bands(
    x: np.ndarray,
    grid: np.ndarray,
    n_iter: int = 28,
    n_boot: int = 90,
    seed: int = 0,
) -> list[tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """Bootstrap uncertainty bands for every EM iteration, not only for the final fit."""
    rng = np.random.default_rng(seed)
    n = len(x)
    densities_by_iter: list[list[np.ndarray]] = [[] for _ in range(n_iter + 1)]
    for _ in range(n_boot):
        sample = rng.choice(x, size=n, replace=True)
        sample_trace = em_gmm_1d_trace(sample, n_iter=n_iter)
        for t, (weights, means, sigmas) in enumerate(sample_trace):
            densities_by_iter[t].append(mixture_pdf(grid, weights, means, sigmas))

    bands: list[tuple[np.ndarray, np.ndarray, np.ndarray]] = []
    for densities in densities_by_iter:
        arr = np.vstack(densities)
        bands.append((
            np.quantile(arr, 0.05, axis=0),
            np.mean(arr, axis=0),
            np.quantile(arr, 0.95, axis=0),
        ))
    return bands


def make_mixture_gif() -> Path:
    x = geyser["duration"].dropna().to_numpy(float)
    grid = np.linspace(x.min() - 0.5, x.max() + 0.5, 360)
    bins = np.linspace(grid.min(), grid.max(), 30)
    trace = em_gmm_1d_trace(x, n_iter=28)
    # 修正点: 最終iterationの帯を全フレームで使い回さず、各iteration時点の帯を事前計算する。
    bands_by_iter = gmm_bootstrap_trace_bands(x, grid, n_iter=len(trace) - 1, n_boot=90, seed=333)
    band_area_by_iter = [float(np.trapezoid(hi - lo, grid)) for lo, _mid, hi in bands_by_iter]
    print(
        "mixture bootstrap band area by iteration: "
        f"first={band_area_by_iter[0]:.3f}, "
        f"middle={band_area_by_iter[len(band_area_by_iter)//2]:.3f}, "
        f"last={band_area_by_iter[-1]:.3f}"
    )

    frames: list[np.ndarray] = []
    max_band_height = max(float(hi.max()) for lo, mid, hi in bands_by_iter)
    max_trace_height = max(float(mixture_pdf(grid, weights, means, sigmas).max()) for weights, means, sigmas in trace)
    y_top = max(max_band_height, max_trace_height) * 1.25
    for t, (weights, means, sigmas) in enumerate(trace):
        lo, mid, hi = bands_by_iter[t]
        pdf = mixture_pdf(grid, weights, means, sigmas)
        comps = [weights[k] * norm.pdf(grid, means[k], sigmas[k]) for k in range(2)]

        fig, ax = plt.subplots(figsize=(8.5, 5.2), constrained_layout=True)
        ax.hist(x, bins=bins, density=True, color="0.76", alpha=0.45, label="噴出時間のヒストグラム")
        ax.fill_between(grid, lo, hi, color="#8172B3", alpha=0.20, label="このiterationの90%帯")
        ax.plot(grid, pdf, color="#CCB974", lw=3.0, label="混合密度")
        ax.plot(grid, comps[0], color="#C44E52", lw=1.8, ls="--", label="成分1")
        ax.plot(grid, comps[1], color="#4C72B0", lw=1.8, ls="--", label="成分2")
        ax.set_title("3. 多峰性分布への 2 成分混合正規フィッティング")
        ax.set_xlabel("geyser duration")
        ax.set_ylabel("density")
        ax.set_xlim(grid.min(), grid.max())
        ax.set_ylim(0, y_top)
        add_note(
            ax,
            f"EM iteration {t}/{len(trace)-1}\nπ={weights.round(2)}\nμ={means.round(2)}\nσ={sigmas.round(2)}\n90%帯面積={band_area_by_iter[t]:.3f}",
        )
        ax.legend(loc="upper right", frameon=True)
        frames.append(fig_to_frame(fig))
        plt.close(fig)

    return save_gif(frames, GIF_DIR / "03_gaussian_mixture_em_fit.gif", fps=7)


gif_03 = make_mixture_gif()
gif_03


mixture bootstrap band area by iteration: first=0.069, middle=0.452, last=0.451


saved D:\dev\dev_statistics\outputs\gifs\03_gaussian_mixture_em_fit.gif (29 frames, 7 fps)


WindowsPath('D:/dev/dev_statistics/outputs/gifs/03_gaussian_mixture_em_fit.gif')

## 4. カテゴリありデータに対する部分プーリング線形回帰

"
    "- **データ**: Palmer Penguins の `species` ごとに、`flipper_length_mm` から `body_mass_g` を予測する。
"
    "- **モデルの直感**: 種別ごとの切片・傾きを、全体回帰線からのズレとして持つ。ズレに正則化をかけると、少ないカテゴリほど全体平均へ縮む。
"
    "- **ここでの実装**: 階層ベイズの直感を、次の正則化付き最小二乗で軽量に再現する。

"
    "$$
"
    "y_i = \alpha + \beta x_i + a_{g[i]} + b_{g[i]}x_i + \epsilon_i,
"
    "\quad \lambda \sum_g (a_g^2 + b_g^2)
"
    "$$

"
    "- **動き**: $\lambda$ が非常に大きいと全カテゴリが同じ線（完全プーリング）になり、$\lambda$ を下げるとカテゴリ別の線が部分的に出てくる。
"
    "- **不確実性**: 各 $\lambda$ で層化ブートストラップし、カテゴリ別予測線の 90% 帯を塗る。
"


In [7]:
def build_partial_pooling_design(x_std: np.ndarray, group_idx: np.ndarray, n_groups: int) -> np.ndarray:
    n = len(x_std)
    X = np.zeros((n, 2 + 2 * n_groups), dtype=float)
    X[:, 0] = 1.0
    X[:, 1] = x_std
    for g in range(n_groups):
        mask = group_idx == g
        X[mask, 2 + g] = 1.0
        X[mask, 2 + n_groups + g] = x_std[mask]
    return X


def fit_partial_pooling_ridge(x_std: np.ndarray, y: np.ndarray, group_idx: np.ndarray, n_groups: int, lam: float) -> np.ndarray:
    X = build_partial_pooling_design(x_std, group_idx, n_groups)
    penalty = np.zeros(X.shape[1])
    penalty[2:] = lam
    lhs = X.T @ X + np.diag(penalty)
    rhs = X.T @ y
    return np.linalg.solve(lhs, rhs)


def predict_partial_pooling(beta: np.ndarray, x_grid_std: np.ndarray, group_id: int, n_groups: int) -> np.ndarray:
    gidx = np.full(len(x_grid_std), group_id, dtype=int)
    Xp = build_partial_pooling_design(x_grid_std, gidx, n_groups)
    return Xp @ beta


def stratified_bootstrap_indices(groups: np.ndarray, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    idx_parts = []
    for g in np.unique(groups):
        idx = np.flatnonzero(groups == g)
        idx_parts.append(rng.choice(idx, size=len(idx), replace=True))
    return np.concatenate(idx_parts)


def partial_pooling_bootstrap_bands(
    x_std: np.ndarray,
    y: np.ndarray,
    group_idx: np.ndarray,
    x_grid_std: np.ndarray,
    n_groups: int,
    lam: float,
    n_boot: int = 70,
    seed: int = 0,
) -> dict[int, tuple[np.ndarray, np.ndarray]]:
    preds_by_group = {g: [] for g in range(n_groups)}
    for b in range(n_boot):
        idx = stratified_bootstrap_indices(group_idx, seed + b)
        beta_b = fit_partial_pooling_ridge(x_std[idx], y[idx], group_idx[idx], n_groups, lam=lam)
        for g in range(n_groups):
            preds_by_group[g].append(predict_partial_pooling(beta_b, x_grid_std, g, n_groups))
    bands = {}
    for g, preds in preds_by_group.items():
        arr = np.vstack(preds)
        bands[g] = (np.quantile(arr, 0.05, axis=0), np.quantile(arr, 0.95, axis=0))
    return bands


def make_partial_pooling_gif() -> Path:
    df = penguins[["species", "flipper_length_mm", "body_mass_g"]].dropna().copy()
    df["body_mass_kg"] = df["body_mass_g"] / 1000.0
    groups = sorted(df["species"].unique())
    group_to_idx = {g: i for i, g in enumerate(groups)}
    group_idx = df["species"].map(group_to_idx).to_numpy(int)
    x_raw = df["flipper_length_mm"].to_numpy(float)
    y = df["body_mass_kg"].to_numpy(float)
    x_mean, x_std_scale = x_raw.mean(), x_raw.std(ddof=1)
    x_std = (x_raw - x_mean) / x_std_scale
    x_grid = np.linspace(x_raw.min() - 3, x_raw.max() + 3, 220)
    x_grid_std = (x_grid - x_mean) / x_std_scale

    lambdas = np.geomspace(1e5, 12.0, 30)
    palette = dict(zip(groups, sns.color_palette("Set2", n_colors=len(groups))))

    # 比較用: 完全プーリング線
    pooled_beta = fit_ols(x_raw, y)
    pooled_pred = pooled_beta[0] + pooled_beta[1] * x_grid

    frames: list[np.ndarray] = []
    for i, lam in enumerate(lambdas):
        beta = fit_partial_pooling_ridge(x_std, y, group_idx, len(groups), lam=lam)
        bands = partial_pooling_bootstrap_bands(
            x_std, y, group_idx, x_grid_std, len(groups), lam=lam, n_boot=55, seed=4000 + 100 * i
        )

        fig, ax = plt.subplots(figsize=(8.7, 5.4), constrained_layout=True)
        for g_name in groups:
            g = group_to_idx[g_name]
            mask = group_idx == g
            color = palette[g_name]
            ax.scatter(
                x_raw[mask],
                y[mask],
                s=34,
                alpha=0.65,
                color=color,
                edgecolor="white",
                linewidth=0.4,
                label=f"{g_name} data",
            )
            pred = predict_partial_pooling(beta, x_grid_std, g, len(groups))
            lo, hi = bands[g]
            ax.fill_between(x_grid, lo, hi, color=color, alpha=0.13)
            ax.plot(x_grid, pred, color=color, lw=2.4, label=f"{g_name} partial")

        ax.plot(x_grid, pooled_pred, color="0.25", lw=1.7, ls=":", label="完全プーリング線")
        ax.set_title("4. カテゴリ別線形回帰: 完全プーリング → 部分プーリング")
        ax.set_xlabel("flipper_length_mm")
        ax.set_ylabel("body_mass_kg")
        ax.set_xlim(x_grid.min(), x_grid.max())
        ax.set_ylim(y.min() - 0.35, y.max() + 0.35)
        pooling_label = "ほぼ完全プーリング" if lam > 2e4 else ("部分プーリング" if lam > 20 else "カテゴリ差がかなり出る")
        add_note(
            ax,
            f"正則化 λ={lam:,.1f}\n状態: {pooling_label}\n色の帯 = 90% ブートストラップ帯",
        )
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), borderaxespad=0.0, frameon=True, fontsize=8)
        frames.append(fig_to_frame(fig))
        plt.close(fig)

    return save_gif(frames, GIF_DIR / "04_partial_pooling_regression.gif", fps=7)


gif_04 = make_partial_pooling_gif()
gif_04


saved D:\dev\dev_statistics\outputs\gifs\04_partial_pooling_regression.gif (30 frames, 7 fps)


WindowsPath('D:/dev/dev_statistics/outputs/gifs/04_partial_pooling_regression.gif')

## 5. 作成したGIF

"
    "ノートを実行すると、以下のファイルが生成される。

"
    "1. `outputs/gifs/01_normal_distribution_fit.gif`
"
    "2. `outputs/gifs/02_linear_regression_fit.gif`
"
    "3. `outputs/gifs/03_gaussian_mixture_em_fit.gif`
"
    "4. `outputs/gifs/04_partial_pooling_regression.gif`

"
    "Jupyter 上では次のセルでまとめて表示する。
"


In [8]:
from IPython.display import HTML, display

items = [
    ("1. 分布フィッティング", gif_01),
    ("2. 線形回帰", gif_02),
    ("3. 混合モデル", gif_03),
    ("4. 部分プーリング回帰", gif_04),
]
html = ["<div style='display:grid; grid-template-columns: 1fr; gap: 24px;'>"]
for title, path in items:
    rel = path.relative_to(PROJECT_ROOT).as_posix()
    html.append(f"<section><h3>{title}</h3><p><code>{rel}</code></p><img src='../{rel}' style='max-width: 920px; width: 100%; border:1px solid #ddd;'></section>")
html.append("</div>")
display(HTML("\n".join(html)))


## 6. まとめと発展

"
    "- **1 → 2**: 分布だけを見る段階から、説明変数を使って平均構造を説明する段階へ進んだ。
"
    "- **2 → 3**: 単一の山では説明しづらい分布に対し、潜在クラスを仮定する混合モデルを導入した。
"
    "- **3 → 4**: カテゴリ別に完全に別モデルを作るのではなく、全体情報とカテゴリ情報をバランスさせる部分プーリングを導入した。

"
    "発展案:

"
    "- PyMC/Stan で実際の事後分布から posterior predictive interval を描く。
"
    "- `tips` データで曜日・時間帯ごとのチップ率を階層モデル化する。
"
    "- `mpg` データで `origin` 別に非線形回帰・部分プーリングを行う。
"
    "- GIF だけでなく、`matplotlib.animation` や `plotly` による HTML アニメーションへ拡張する。
"
